# 📚 Notebook 02 — Technical Indicators from Scratch

**Phase 1: Foundations** · Prerequisites: Notebook 01 (OHLCV data)

---

## 🎯 Learning Objectives

After this notebook, you will be able to:

1. Compute **Exponential Moving Averages (EMA)** and understand the smoothing parameter $\alpha$
2. Build **RSI** (Relative Strength Index) using the EWM method — exactly as the production bot does
3. Construct **Bollinger Bands** and interpret band width as a volatility measure
4. Calculate **realized volatility** from log returns
5. Implement **MACD** (Moving Average Convergence/Divergence)
6. Understand how each indicator maps to a production signal module

In [ ]:
# ── Environment Setup ──
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import requests

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# ── Load sample data ──
def fetch_btc_hourly(days: int = 60) -> pd.DataFrame:
    """Fetch BTC hourly data for the course."""
    end_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_ms = end_ms - days * 86_400_000
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        resp = requests.get("https://api.binance.com/api/v3/klines",
                           params={"symbol": "BTCUSDT", "interval": "1h",
                                   "startTime": cursor, "endTime": end_ms, "limit": 1000},
                           timeout=10)
        resp.raise_for_status()
        rows = resp.json()
        if not rows: break
        all_rows.extend(rows)
        cursor = int(rows[-1][0]) + 3_600_000
        if len(rows) < 1000: break
    df = pd.DataFrame(all_rows, columns=["open_time","open","high","low","close","volume",
                                          "close_time","quote_vol","trades","taker_base","taker_quote","ignore"])
    for col in ["open","high","low","close","volume","quote_vol"]:
        df[col] = df[col].astype(float)
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df = df.set_index("timestamp")[["open","high","low","close","volume","quote_vol"]]
    return df

df = fetch_btc_hourly(60)
close = df["close"]
print(f"✅ Loaded {len(df)} hourly candles: {df.index[0].date()} → {df.index[-1].date()}")

---

## 📐 Section 1: Exponential Moving Average (EMA)

A moving average smooths noisy price data to reveal the underlying trend.

### Simple Moving Average (SMA) — Baseline

$$\text{SMA}_k = \frac{1}{k} \sum_{i=0}^{k-1} P_{t-i}$$

All $k$ prices are weighted equally. Problem: SMA is **laggy** — it takes $k$ periods to fully reflect a price change.

### EMA — Exponentially Weighted

$$\text{EMA}_t = \alpha \cdot P_t + (1 - \alpha) \cdot \text{EMA}_{t-1}$$

where the smoothing factor is:

$$\alpha = \frac{2}{k + 1}$$

| $k$ | $\alpha$ | Half-life | Behavior |
|-----|----------|-----------|----------|
| 10  | 0.182    | ~6 bars   | Fast, noisy |
| 20  | 0.095    | ~13 bars  | Medium (our bot's fast EMA) |
| 50  | 0.039    | ~34 bars  | Slow, smooth (our bot's slow EMA) |

The bot uses **EMA-20** and **EMA-50** for regime detection (see Notebook 06).

In [ ]:
# ── EMA from scratch, then with pandas ──

# Method 1: Manual loop (to understand the math)
def ema_manual(prices: pd.Series, span: int) -> pd.Series:
    """Compute EMA step by step using the recursive formula."""
    alpha = 2.0 / (span + 1)
    result = np.empty(len(prices))
    result[0] = prices.iloc[0]  # Initialize with first price
    for i in range(1, len(prices)):
        result[i] = alpha * prices.iloc[i] + (1 - alpha) * result[i - 1]
    return pd.Series(result, index=prices.index, name=f"EMA_{span}")

# Method 2: Pandas built-in (production)
def ema_pandas(prices: pd.Series, span: int) -> pd.Series:
    """EMA using pandas' optimized ewm() — the production approach."""
    return prices.ewm(span=span, adjust=False).mean()

# Compare both methods
ema20_manual = ema_manual(close, 20)
ema20_pandas = ema_pandas(close, 20)

# They should be identical
max_diff = (ema20_manual - ema20_pandas).abs().max()
print(f"Max difference between manual and pandas EMA: {max_diff:.2e}")
print(f"(Should be ~0 or floating-point epsilon)")

In [ ]:
# ── Visualize EMA crossovers ──
ema20 = ema_pandas(close, 20)
ema50 = ema_pandas(close, 50)

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(close.index, close, label='Close', alpha=0.5, linewidth=0.8)
ax.plot(ema20.index, ema20, label='EMA-20 (fast)', linewidth=2, color='#2196F3')
ax.plot(ema50.index, ema50, label='EMA-50 (slow)', linewidth=2, color='#FF9800')

# Mark crossover points
cross_up = (ema20 > ema50) & (ema20.shift(1) <= ema50.shift(1))
cross_down = (ema20 < ema50) & (ema20.shift(1) >= ema50.shift(1))
ax.scatter(close.index[cross_up], close[cross_up], marker='^', c='green', s=100, zorder=5, label='Bullish cross')
ax.scatter(close.index[cross_down], close[cross_down], marker='v', c='red', s=100, zorder=5, label='Bearish cross')

ax.set_title('BTC/USDT — EMA Crossovers (the basis of Regime Detection)', fontsize=14)
ax.set_ylabel('Price (USDT)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n📌 The bot's regime_detector.py uses exactly this EMA-20/EMA-50 crossover")
print(f"   EMA-20 > EMA-50 → Bullish signal")
print(f"   EMA-20 < EMA-50 → Bearish signal")

---

## 📐 Section 2: RSI (Relative Strength Index)

RSI measures **momentum** on a 0–100 scale. It answers: *"How strong are recent gains vs. recent losses?"*

### The Math

1. Compute price changes: $\Delta_t = P_t - P_{t-1}$
2. Separate gains and losses:
   - $G_t = \max(\Delta_t, 0)$
   - $L_t = \max(-\Delta_t, 0)$
3. Smooth with EWM:
   - $\overline{G} = \text{EWM}(G, \text{span}=k)$
   - $\overline{L} = \text{EWM}(L, \text{span}=k)$
4. Relative Strength: $RS = \overline{G} / \overline{L}$
5. RSI: $\text{RSI} = 100 - \frac{100}{1 + RS}$

### Interpretation

| RSI Range | Market State | What the Bot Does |
|-----------|-------------|------------------|
| > 70 | Overbought | Momentum signal reduces weight |
| 30–70 | Neutral | Normal operation |
| < 30 | Oversold | Mean reversion signal activates |
| < 45 | Our bot's threshold | `momentum.py` filters out weak assets |

In [ ]:
# ── RSI from scratch ──

def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    """Calculate RSI using the EWM method.
    
    This is the EXACT same approach used in bot/signals/momentum.py:
    calculate_rsi(close_prices, period=14)
    """
    # Step 1: Price changes
    delta = prices.diff()
    
    # Step 2: Separate gains and losses
    gains = delta.clip(lower=0)      # Only keep positive changes
    losses = (-delta).clip(lower=0)  # Absolute value of negative changes
    
    # Step 3: Exponentially weighted moving average
    avg_gain = gains.ewm(span=period, adjust=False).mean()
    avg_loss = losses.ewm(span=period, adjust=False).mean()
    
    # Step 4: Relative Strength
    rs = avg_gain / avg_loss
    
    # Step 5: RSI formula
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

rsi_14 = calculate_rsi(close, 14)

print(f"Current RSI(14): {rsi_14.iloc[-1]:.1f}")
print(f"Min RSI: {rsi_14.min():.1f}")
print(f"Max RSI: {rsi_14.max():.1f}")

In [ ]:
# ── Compare with production RSI ──
from bot.signals.momentum import calculate_rsi as prod_rsi

rsi_prod = prod_rsi(close, 14)
max_diff = (rsi_14 - rsi_prod).abs().max()
print(f"Max difference from production RSI: {max_diff:.2e}")
print("✅ Our implementation matches the production code!")

In [ ]:
# ── RSI visualization with overbought/oversold zones ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[2, 1], sharex=True)

# Price
ax1.plot(close.index, close, label='BTC Close', color='#333', linewidth=1)
ax1.set_ylabel('Price (USDT)')
ax1.set_title('BTC/USDT Price and RSI(14)', fontsize=14)
ax1.legend()

# RSI
ax2.plot(rsi_14.index, rsi_14, label='RSI(14)', color='#9C27B0', linewidth=1.2)
ax2.axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
ax2.axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
ax2.axhline(45, color='orange', linestyle=':', alpha=0.7, label='Bot threshold (45)')
ax2.fill_between(rsi_14.index, 70, 100, alpha=0.1, color='red')
ax2.fill_between(rsi_14.index, 0, 30, alpha=0.1, color='green')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

---

## 📐 Section 3: Bollinger Bands

Bollinger Bands create a **dynamic envelope** around price, expanding with volatility and contracting during calm periods.

### The Math

$$\text{Middle Band} = \text{SMA}_k(P)$$

$$\text{Upper Band} = \text{SMA}_k + n_\sigma \cdot \sigma_k$$

$$\text{Lower Band} = \text{SMA}_k - n_\sigma \cdot \sigma_k$$

where $\sigma_k$ is the rolling standard deviation of the last $k$ prices.

Our bot uses $k = 20$ and $n_\sigma = 2.0$ (from `config/strategy_params.yaml`).

### Key Signals

- **Price touches lower band** → potential **oversold** (mean reversion buy signal)
- **Price touches upper band** → potential **overbought**
- **Band squeeze** (bands narrow) → low volatility, breakout may follow
- **Band expansion** → high volatility, trend in progress

In [ ]:
# ── Bollinger Bands from scratch ──

def bollinger_bands(prices: pd.Series, period: int = 20, num_std: float = 2.0):
    """Compute Bollinger Bands.
    
    Production reference: bot/signals/mean_reversion.py → build_mean_reversion_frame()
    Config: strategy_params.yaml → mean_reversion.bb_period=20, bb_std=2.0
    """
    middle = prices.rolling(period).mean()
    std = prices.rolling(period).std()
    upper = middle + num_std * std
    lower = middle - num_std * std
    
    # Band width: normalized volatility measure
    band_width = (upper - lower) / middle
    
    # %B: Where is price within the bands? (0 = lower, 1 = upper)
    pct_b = (prices - lower) / (upper - lower)
    
    return pd.DataFrame({
        'middle': middle,
        'upper': upper,
        'lower': lower,
        'band_width': band_width,
        'pct_b': pct_b,
    }, index=prices.index)

bb = bollinger_bands(close, 20, 2.0)
print(f"Current %B: {bb['pct_b'].iloc[-1]:.3f}")
print(f"  0.0 = at lower band, 1.0 = at upper band")
print(f"  Current band width: {bb['band_width'].iloc[-1]:.4f}")

In [ ]:
# ── Bollinger Bands visualization ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[3, 1], sharex=True)

# Price with bands
ax1.plot(close.index, close, label='Close', color='#333', linewidth=1)
ax1.plot(bb['middle'].index, bb['middle'], label='SMA(20)', color='#2196F3', linewidth=1)
ax1.plot(bb['upper'].index, bb['upper'], label='Upper Band (+2σ)', color='red', linewidth=0.8, linestyle='--')
ax1.plot(bb['lower'].index, bb['lower'], label='Lower Band (-2σ)', color='green', linewidth=0.8, linestyle='--')
ax1.fill_between(bb.index, bb['upper'], bb['lower'], alpha=0.1, color='blue')

# Highlight touches
touch_lower = close <= bb['lower']
touch_upper = close >= bb['upper']
ax1.scatter(close.index[touch_lower], close[touch_lower], c='green', s=30, zorder=5, label='Touch lower')
ax1.scatter(close.index[touch_upper], close[touch_upper], c='red', s=30, zorder=5, label='Touch upper')

ax1.set_title('BTC/USDT — Bollinger Bands (20, 2.0)', fontsize=14)
ax1.set_ylabel('Price (USDT)')
ax1.legend(fontsize=9)

# Band width
ax2.fill_between(bb.index, 0, bb['band_width'], alpha=0.3, color='purple')
ax2.plot(bb['band_width'].index, bb['band_width'], color='purple', linewidth=1)
ax2.set_ylabel('Band Width')
ax2.set_title('Bollinger Band Width (volatility measure)', fontsize=11)

plt.tight_layout()
plt.show()

---

## 📐 Section 4: Realized Volatility

**Volatility** is the intensity of price fluctuations. The bot uses it for:

- **Regime detection**: High volatility → bear/ranging regime
- **Risk budgeting**: Scale position sizes inversely with volatility

### The Math

1. Log returns: $r_t = \ln(P_t / P_{t-1})$
2. Rolling standard deviation: $\sigma_k = \text{std}(r_{t-k+1}, \ldots, r_t)$
3. Annualize: $\sigma_{\text{ann}} = \sigma_k \cdot \sqrt{N}$ where $N = 8{,}760$ for hourly data (365 × 24)

The bot's regime detector uses a **volatility threshold** to distinguish calm from turbulent markets.

In [ ]:
# ── Realized volatility from scratch ──

# Step 1: Log returns
log_returns = np.log(close / close.shift(1))

# Step 2: Rolling volatility (20-period)
vol_20 = log_returns.rolling(20).std()

# Step 3: Annualize (hourly data → 8760 hours/year)
vol_annualized = vol_20 * np.sqrt(8760)

print(f"Current hourly vol: {vol_20.iloc[-1]:.6f}")
print(f"Annualized vol:     {vol_annualized.iloc[-1]:.1%}")
print(f"Mean annualized:    {vol_annualized.mean():.1%}")

# The bot's regime_detector.py compares this to a threshold (default: 0.02 hourly)
VOL_THRESHOLD = 0.02
print(f"\nBot vol threshold: {VOL_THRESHOLD}")
print(f"Current vs threshold: {'HIGH vol ⚠️' if vol_20.iloc[-1] > VOL_THRESHOLD else 'Normal vol ✅'}")

In [ ]:
# ── Volatility visualization ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), height_ratios=[2, 1], sharex=True)

ax1.plot(close.index, close, color='#333', linewidth=1)
ax1.set_ylabel('Price (USDT)')
ax1.set_title('BTC/USDT — Price and Rolling Volatility', fontsize=14)

ax2.fill_between(vol_20.index, 0, vol_20, alpha=0.3, color='orange')
ax2.plot(vol_20.index, vol_20, color='darkorange', linewidth=1)
ax2.axhline(VOL_THRESHOLD, color='red', linestyle='--', label=f'Regime threshold ({VOL_THRESHOLD})')
ax2.set_ylabel('Hourly Vol (σ₂₀)')
ax2.legend()

plt.tight_layout()
plt.show()

---

## 📐 Section 5: MACD (Moving Average Convergence/Divergence)

MACD combines two EMAs to detect **trend changes and momentum shifts**.

### The Math

$$\text{MACD line} = \text{EMA}_{12}(P) - \text{EMA}_{26}(P)$$

$$\text{Signal line} = \text{EMA}_9(\text{MACD line})$$

$$\text{Histogram} = \text{MACD line} - \text{Signal line}$$

### Signals

- **MACD crosses above signal** → Bullish momentum
- **MACD crosses below signal** → Bearish momentum
- **Histogram growing** → Trend strengthening
- **Histogram shrinking** → Trend weakening

In [ ]:
# ── MACD from scratch ──

def compute_macd(prices: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    """Compute MACD line, signal line, and histogram."""
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    
    return pd.DataFrame({
        'macd': macd_line,
        'signal': signal_line,
        'histogram': histogram,
    }, index=prices.index)

macd = compute_macd(close)
print(f"Current MACD: {macd['macd'].iloc[-1]:.2f}")
print(f"Signal:       {macd['signal'].iloc[-1]:.2f}")
print(f"Histogram:    {macd['histogram'].iloc[-1]:.2f}")

In [ ]:
# ── MACD visualization ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[2, 1], sharex=True)

ax1.plot(close.index, close, color='#333', linewidth=1)
ax1.set_ylabel('Price (USDT)')
ax1.set_title('BTC/USDT — MACD', fontsize=14)

ax2.plot(macd['macd'].index, macd['macd'], label='MACD', color='#2196F3', linewidth=1.5)
ax2.plot(macd['signal'].index, macd['signal'], label='Signal', color='#FF9800', linewidth=1.5)
colors = ['#26a69a' if v >= 0 else '#ef5350' for v in macd['histogram']]
ax2.bar(macd.index, macd['histogram'], color=colors, alpha=0.5, width=0.03)
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.legend()
ax2.set_ylabel('MACD')

plt.tight_layout()
plt.show()

---

## 💻 Section 6: Production Indicator Frame

The bot's `mean_reversion.py` builds a complete indicator DataFrame in one call. Let's use it and see how all indicators work together:

In [ ]:
# ── Production mean reversion indicator frame ──
from bot.signals.mean_reversion import build_mean_reversion_frame

# The production function computes RSI + Bollinger %B in one vectorized pass
mr_frame = build_mean_reversion_frame(close, rsi_period=14, bb_period=20, bb_std=2.0)

print("Columns in the production indicator frame:")
print(mr_frame.columns.tolist())
print(f"\nLatest values:")
mr_frame.tail(3)

---

## 📐 Summary: Indicator → Bot Module Mapping

| Indicator | Formula | Bot Module | Purpose |
|-----------|---------|------------|----------|
| EMA-20/50 | $\alpha P_t + (1-\alpha) \text{EMA}_{t-1}$ | `regime_detector.py` | Bull/Bear classification |
| RSI(14) | $100 - 100/(1+RS)$ | `momentum.py` | Asset strength filter |
| Bollinger %B | $(P - L) / (U - L)$ | `mean_reversion.py` | Oversold detection |
| Volatility | $\text{std}(\ln P_t/P_{t-1})$ | `regime_detector.py` | High-vol regime flag |
| MACD | $\text{EMA}_{12} - \text{EMA}_{26}$ | Supplementary | Trend confirmation |

---

## 🔬 Exercises

### Exercise 1: Multi-Asset RSI Dashboard 🔬

Fetch hourly data for 3 assets (BTC, ETH, SOL), compute RSI(14) for each, and plot them on the same chart. Which asset is currently most overbought/oversold?

In [ ]:
# ── Exercise 1: Your code here ──

# YOUR CODE HERE

### Exercise 2: Band Width Percentile ⭐

Compute the current Bollinger Band width as a percentile of the last 60 days. A very low percentile (< 10%) indicates a "squeeze" — a potential breakout is coming.

In [ ]:
# ── Exercise 2: Your code here ──

# Hint: Use scipy.stats.percentileofscore()
# YOUR CODE HERE

### Exercise 3: Custom Indicator ⭐⭐

Create a composite indicator that combines RSI and Bollinger %B into a single score from -1 (very oversold) to +1 (very overbought). Test it on BTC data.

In [ ]:
# ── Exercise 3: Your code here ──

# YOUR CODE HERE

---

## ✅ Knowledge Check

1. Why does EMA react faster than SMA to price changes?
2. What RSI value means "perfectly balanced" between recent gains and losses?
3. If Bollinger Bands are narrowing, what does that tell you about volatility?
4. Why do we use **log returns** instead of simple returns for volatility?
5. What are the three EMAs in a standard MACD setup?

<details>
<summary>Click for answers</summary>

1. EMA gives more weight to recent prices (exponentially decaying weights), so new data has more influence
2. RSI = 50 means average gains equal average losses
3. Volatility is decreasing — a "squeeze" that often precedes a breakout move
4. Log returns are additive across time and approximately normally distributed, making statistical analysis more rigorous
5. EMA-12 (fast), EMA-26 (slow), and EMA-9 (signal line)

</details>

---

## 🔗 Next: Notebook 03 — Risk Metrics Deep Dive

You now know how to measure what the market is doing. The next step is measuring **how well your strategy performs** — using Sharpe, Sortino, and Calmar ratios. In Notebook 03, we'll derive these metrics from first principles, including the Delta method for standard error estimation.

**Open:** `03_Risk_Metrics_Deep_Dive.ipynb`